<a href="https://colab.research.google.com/github/zinebaknin-png/Presentation-DrTeou-Projet-fin-bootcamp-LeWagon/blob/main/DrT%C3%A9ou.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Evolution temporelle des déserts médicaux en France et projections**

**Import Bibliothèque**

In [1]:
# Chargement des librairies
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

**Important: Il existe un département "999" aka "Tout département" pour différentes régions ultramarines**

In [3]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()

In [ ]:
tables = list(client.list_tables("zinebakninsql.Teou"))
for t in tables:
    print(t.table_id)

In [ ]:
client = bigquery.Client(project="zinebakninsql")

**import** **tables**

In [ ]:

DROM = ["971", "972", "973", "974", "975", "976", "999"]

df_densite = client.query("""
SELECT *
FROM `zinebakninsql.Teou.densite`
""").to_dataframe()

# Exclusion DROM une fois pour toutes
df_densite = df_densite[
    (~df_densite["departement"].astype(str).isin(DROM)) &
    (df_densite["libelle_departement"] != "Tout département")
].copy()

In [ ]:
df_densite_patientele = client.query("""
SELECT *
FROM `zinebakninsql.Teou.densite_patientele`
""").to_dataframe()

In [ ]:
df_effectif_densite = client.query("""
SELECT *
FROM `zinebakninsql.Teou.effectif_densite`
""").to_dataframe()

In [ ]:
df_effectif_densite_clean = client.query("""
SELECT *
FROM `zinebakninsql.Teou.effectif_densite_clean`
""").to_dataframe()

In [ ]:
df_patientele = client.query("""
SELECT *
FROM `zinebakninsql.Teou.patientele`
""").to_dataframe()

In [ ]:
print(df_densite.shape)
print(df_densite_patientele.shape)
print(df_effectif_densite.shape)
print(df_effectif_densite_clean.shape)
print(df_patientele.shape)

**stats clés**

In [ ]:
import pandas as pd
stats = pd.DataFrame({
    "Indicateur": [
        "Nombre de professions analysées",
        "Nombre de départements",
        "Période couverte"
    ],
    "Valeur": [
        df_effectif_densite_clean["profession_sante"].nunique(),
        df_effectif_densite_clean["departement"].nunique(),
        f"{df_effectif_densite_clean['annee'].min()} - {df_effectif_densite_clean['annee'].max()}"
    ]
})

stats.style.set_properties(**{"text-align": "left"}).hide(axis="index")

**Exagone VS DROM**

In [ ]:
# Comparaison exagone VS DROM
drom = ["971", "972", "973", "974", "976"]
df_densite["zone"] = df_densite["departement"].apply(
    lambda x: "DROM" if x in drom else "Métropole"
)
fig = px.box(df_densite, x="zone", y="densite", color="zone",
             title="Densité médicale : Métropole vs DROM")


Nombre de professionnels de santé par département (France 2024)

In [ ]:
import pandas as pd
import plotly.express as px
import requests

# 1. dernière année
annee_max = df_effectif_densite_clean["annee"].max()

df_filtered = df_effectif_densite_clean[
    df_effectif_densite_clean["annee"] == annee_max
].copy()


# 3. agrégation
df_map = (
    df_filtered
    .groupby(["departement", "libelle_departement"], as_index=False)["effectif"]
    .sum()
)

# 4. nettoyage codes département
df_map["departement"] = (
    df_map["departement"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# 5. geojson
geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements.geojson"
geojson = requests.get(geojson_url).json()

# 6. carte
fig = px.choropleth(
    df_map,
    geojson=geojson,
    locations="departement",
    featureidkey="properties.code",
    color="effectif",
    color_continuous_scale="Blues",
    hover_name="libelle_departement",
    hover_data={"effectif": ":,"},
    title=f"Nombre de professionnels de santé (France {annee_max})"
)

fig.update_geos(fitbounds="locations", visible=False)
import geopandas as gpd

# Calculer les centroïdes des régions depuis le geojson
gdf = gpd.GeoDataFrame.from_features(geojson["features"])
gdf["centroid_lon"] = gdf.geometry.centroid.x
gdf["centroid_lat"] = gdf.geometry.centroid.y

# Ajouter les noms des régions sur la carte
fig.add_scattermapbox(
    lat=gdf["centroid_lat"],
    lon=gdf["centroid_lon"],
    mode="text",
    text=gdf["nom"],
    textfont=dict(size=10, color="black"),
    hoverinfo="none"
)
fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(
        len=0.75,
        thickness=15,
        yanchor="middle",
        y=0.5
    )
)

fig.show()

**Nombre de médecins généralistes par département (France 2024)**

In [ ]:
import pandas as pd
import plotly.express as px
import requests

# 1. Filtrer médecins traitants (généralistes)
categories = [
    "Médecins généralistes (hors médecins à expertise particulière - MEP)",
   ]

df_filtered = df_effectif_densite_clean[
    df_effectif_densite_clean["profession_sante"].isin(categories)
]

# 2. Prendre la dernière année
annee_max = df_filtered["annee"].max()

df_filtered = df_filtered[
    df_filtered["annee"] == annee_max
]

# 3. Agrégation par département
df_map = (
    df_filtered
    .groupby(["departement", "libelle_departement"], as_index=False)["effectif"]
    .sum()
)

# 4. Nettoyage codes département
df_map["departement"] = (
    df_map["departement"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# 5. GeoJSON France + DROM-COM
geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements.geojson"
geojson = requests.get(geojson_url).json()

# 6. Carte choroplèthe
fig = px.choropleth(
    df_map,
    geojson=geojson,
    locations="departement",
    featureidkey="properties.code",
    color="effectif",
    color_continuous_scale="Blues",
    hover_name="libelle_departement",
    hover_data={"effectif": ":,"},
    title=f"Nombre de médecins généralistes par département(France {annee_max})"
)

# 7. Mise en forme
fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(
        len=0.75,
        thickness=15,
        yanchor="middle",
        y=0.5
    )
)

fig.show()

In [ ]:
df_densite.columns

In [ ]:
df_densite.head()

**Densité de professionnels de santé par département(France 2024)**


In [ ]:
import pandas as pd
import plotly.express as px
import requests

# 1. Dernière année
annee_max = df_densite["annee"].max()

df_map = df_densite[df_densite["annee"] == annee_max].copy()

# 2. IMPORTANT : toutes les professions de santé (densité globale)
df_map = (
    df_map
    .groupby(["departement", "libelle_departement"], as_index=False)
    .agg({"densite": "mean"})
)

# 3. Nettoyage codes département
df_map["departement"] = (
    df_map["departement"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# 4. Carte choroplèthe
fig = px.choropleth(
    df_map,
    geojson=geojson,
    locations="departement",
    featureidkey="properties.code",
    color="densite",
    color_continuous_scale="RdYlGn",
    hover_name="libelle_departement",
    hover_data={"departement": True, "densite": ":.2f"},
    title=f"Densité médical de toutes les professions de santé pour 100 000 habitants (France {annee_max})"
)

# 5. Mise en forme


fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    coloraxis_colorbar=dict(
        len=0.75,
        thickness=15,
        yanchor="middle",
        y=0.5,
        title="Densité"
    )
)
fig.show()


**Densité de médecins généralistes par département(France 2010_2024)**
Zoom sur le Top 3 et Flop 3

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests

#  Filtre médecins généralistes
df = df_densite[
    (df_densite["profession_sante"] == "Médecins généralistes (hors médecins à expertise particulière - MEP)")
].copy()

df_map = df.groupby(["annee", "departement", "libelle_departement"], as_index=False)["densite"].mean()
df_map["departement"] = df_map["departement"].astype(str)

#  Identification des extrêmes pour chaque année pour la surbrillance
extremes_by_year = []

for annee in df_map['annee'].unique():
    temp = df_map[df_map['annee'] == annee].sort_values('densite')
    extremes = pd.concat([temp.head(3), temp.tail(3)])
    extremes_by_year.append(extremes)

df_extremes = pd.concat(extremes_by_year)

#  Création de la carte de base animée avec échelle fixée
fig = px.choropleth_mapbox(
    df_map,
    geojson=geojson,
    locations="departement",
    featureidkey="properties.code",
    color="densite",
    animation_frame="annee",
    color_continuous_scale=['#E60F05', '#FF6B6B', '#FF9999', '#A4C2F4', '#0E4099', '#05204D'],
    range_color=[44.1, 127.5],
    hover_name="libelle_departement",
    hover_data={"departement": True, "densite": ":.2f"},
    mapbox_style="white-bg",
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    opacity=0.85
)

#  Ajout des frontières noires et surbrillance (Via Frames pour l'animation)
for frame in fig.frames:
    annee_frame = int(frame.name)
    df_ext_annee = df_extremes[df_extremes['annee'] == annee_frame]

    # Sélection du geojson spécifique pour ces 6 départements
    codes_ext = df_ext_annee['departement'].tolist()
    sub_geojson = {"type": "FeatureCollection", "features": [f for f in geojson['features'] if f['properties']['code'] in codes_ext]}

    highlight_trace = go.Choroplethmapbox(
        geojson=sub_geojson,
        locations=df_ext_annee['departement'],
        z=df_ext_annee['densite'],
        featureidkey="properties.code",
        colorscale=['#E60F05', '#FF6B6B', '#FF9999', '#A4C2F4', '#0E4099', '#05204D'],
        zmin=44.1,
        zmax=127.5,
        marker=dict(line=dict(width=2.5, color='black'), opacity=1.0),
        showscale=False,
        hoverinfo='skip'
    )
    frame.data = frame.data + (highlight_trace,)

# Ajout initial pour la première année affichée
annee_init = df_map['annee'].min()
df_ext_init = df_extremes[df_extremes['annee'] == annee_init]
codes_init = df_ext_init['departement'].tolist()
sub_geojson_init = {"type": "FeatureCollection", "features": [f for f in geojson['features'] if f['properties']['code'] in codes_init]}

fig.add_trace(go.Choroplethmapbox(
    geojson=sub_geojson_init,
    locations=df_ext_init['departement'],
    z=df_ext_init['densite'],
    featureidkey="properties.code",
    colorscale=['#E60F05', '#FF6B6B', '#FF9999', '#A4C2F4', '#0E4099', '#05204D'],
    zmin=44.1,
    zmax=127.5,
    marker=dict(line=dict(width=2.5, color='black'), opacity=1.0),
    showscale=False,
    hoverinfo='skip'
))

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    height=650,
    coloraxis_colorbar=dict(len=0.75, thickness=15, yanchor="middle", y=0.5, title="Densité"),
    sliders=[dict(
        currentvalue=dict(visible=False, prefix=""),
        pad=dict(t=10, b=10),
    )],
    annotations=[dict(
        x=0.2,
        y=0.5,
        xref="paper",
        yref="paper",
        text=f"{annee_init}</b>",
        showarrow=False,
        font=dict(size=28, color="black"),
        xanchor="left",
        yanchor="middle",
    )]
)

for frame in fig.frames:
    frame.layout = go.Layout(
        annotations=[dict(
            x=0.2,
            y=0.5,
            xref="paper",
            yref="paper",
            text=f"<b>{frame.name}</b>",
            showarrow=False,
            font=dict(size=28, color="black"),
            xanchor="left",
            yanchor="middle",
        )]
    )

fig.show(config={"scrollZoom": True})
#fig.write_html("carte_medecins.html", include_plotlyjs="cdn")

**Prédiction densité 2025_2036**


**Prédiction densité toutes professions (2025-2036)**

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import requests
from sklearn.linear_model import LinearRegression

# ============================================================
# 1. DONNÉES HISTORIQUES
# ============================================================
df = df_densite[
    (df_densite["departement"] != "999")
].copy()

df_map = (
    df
    .groupby(["annee", "departement", "libelle_departement"], as_index=False)["densite"]
    .mean()
)

df_map["departement"] = df_map["departement"].astype(str)

# ============================================================
# 2. PRÉDICTIONS 2025-2036
# ============================================================
ANNEE_SPLIT = 2022
predictions = []

for dept, group in df_map.groupby("departement"):
    group = group.sort_values("annee").dropna(subset=["densite"])

    train = group[group["annee"] < ANNEE_SPLIT]
    test  = group[group["annee"] >= ANNEE_SPLIT]

    X_train = train[["annee"]]
    y_train = train["densite"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    annees_futures = pd.DataFrame({"annee": range(2025, 2037)})
    densite_pred = model.predict(annees_futures)

    for annee, dens in zip(annees_futures["annee"], densite_pred):
        predictions.append({
            "annee": annee,
            "departement": dept,
            "libelle_departement": group["libelle_departement"].iloc[0],
            "densite": max(0, dens)
        })

df_pred = pd.DataFrame(predictions)

# ============================================================
# 3. FUSION HISTORIQUE + PRÉDICTIONS
# ============================================================
df_final = pd.concat([df_map, df_pred], ignore_index=True)
df_final = df_final.sort_values("annee")
df_final = df_final[df_final["annee"] >= 2025]
df_final = df_final[~df_final["departement"].isin(["971", "972", "973", "974", "975", "976"])]

# ============================================================
# 4. CARTE ANIMÉE 2025-2036
# ============================================================
geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements.geojson"
geojson = requests.get(geojson_url).json()

fig = px.choropleth_mapbox(
    df_final,
    geojson=geojson,
    locations="departement",
    featureidkey="properties.code",
    color="densite",
    animation_frame="annee",
    color_continuous_scale="Blues",
    hover_name="libelle_departement",
    hover_data={"departement": True, "densite": ":.2f"},
    title="Densité toutes professions (2025-2036) — données + prédictions",
    mapbox_style="white-bg",
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    opacity=0.85,
    range_color=[df_final["densite"].min(), df_final["densite"].max()]
)

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    height=650,
    coloraxis_colorbar=dict(
        len=0.75,
        thickness=15,
        yanchor="middle",
        y=0.5,
        title="Densité"
    )
)

fig.show(config={"scrollZoom": True})

**Prédiction densité médecins géneraliste (2025-2036)**

In [ ]:
import pandas as pd
import plotly.express as px
import requests
from sklearn.linear_model import LinearRegression
import numpy as np
import plotly.graph_objects as go

df_mg = df_densite[
      (df_densite["profession_sante"] == "Médecins généralistes (hors médecins à expertise particulière - MEP)")
].copy()

df_map_mg = (
    df_mg
    .groupby(["annee", "departement", "libelle_departement"], as_index=False)["densite"]
    .mean()
)
df_map_mg["departement"] = df_map_mg["departement"].astype(str)

ANNEE_SPLIT = 2022
predictions_mg = []

for dept, group in df_map_mg.groupby("departement"):
    group = group.sort_values("annee").dropna(subset=["densite"])
    train = group[group["annee"] < ANNEE_SPLIT]
    X_train = train[["annee"]]
    y_train = train["densite"]
    model = LinearRegression()
    model.fit(X_train, y_train)
    annees_futures = pd.DataFrame({"annee": range(2025, 2037)})
    densite_pred = model.predict(annees_futures)
    for annee, dens in zip(annees_futures["annee"], densite_pred):
        predictions_mg.append({
            "annee": annee,
            "departement": dept,
            "libelle_departement": group["libelle_departement"].iloc[0],
            "densite": max(0, dens)
        })

df_pred_mg = pd.DataFrame(predictions_mg)
df_final_mg = pd.concat([df_map_mg, df_pred_mg], ignore_index=True)
df_final_mg = df_final_mg.sort_values("annee")
df_final_mg = df_final_mg[df_final_mg["annee"] >= 2025]

geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements.geojson"
geojson = requests.get(geojson_url).json()

extremes_by_year = []
for annee in df_final_mg['annee'].unique():
    temp = df_final_mg[df_final_mg['annee'] == annee].sort_values('densite')
    extremes = pd.concat([temp.head(3), temp.tail(3)])
    extremes_by_year.append(extremes)
df_extremes = pd.concat(extremes_by_year)

fig = px.choropleth_mapbox(
    df_final_mg,
    geojson=geojson,
    locations="departement",
    featureidkey="properties.code",
    color="densite",
    animation_frame="annee",
    color_continuous_scale=['#E60F05', '#FF6B6B', '#FF9999', '#A4C2F4', '#0E4099', '#05204D'],
    hover_name="libelle_departement",
    hover_data={"departement": True, "densite": ":.2f"},
    mapbox_style="white-bg",
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    opacity=0.85,
    range_color=[27.6, 120.41]  # Échelle figée pour la base
)

for frame in fig.frames:
    annee_f = int(frame.name)
    df_ext = df_extremes[df_extremes['annee'] == annee_f]
    codes = df_ext['departement'].tolist()
    sub_gj = {'type': 'FeatureCollection', 'features': [f for f in geojson['features'] if f['properties']['code'] in codes]}

    highlight = go.Choroplethmapbox(
        geojson=sub_gj, locations=df_ext['departement'], z=df_ext['densite'],
        featureidkey='properties.code',
        colorscale=['#E60F05', '#FF6B6B', '#FF9999', '#A4C2F4', '#0E4099', '#05204D'],
        zmin=27.6, zmax=120.41,  # Échelle figée pour les frames
        showscale=False,
        marker=dict(line=dict(width=1.5, color='black'), opacity=1.0), hoverinfo='skip'
    )
    frame.data = frame.data + (highlight,)

annee_init = df_final_mg['annee'].min()

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0},
    height=650,
    coloraxis_colorbar=dict(len=0.75, thickness=15, yanchor="middle", y=0.5, title="Densité"),
    sliders=[dict(
        currentvalue=dict(visible=False, prefix=""),
        pad=dict(t=10, b=10),
    )],
    annotations=[dict(
        x=0.2, y=0.5, xref="paper", yref="paper",
        text=f"{annee_init}</b>",
        showarrow=False,
        font=dict(size=28, color="black"),
        xanchor="left", yanchor="middle",
    )]
)

for frame in fig.frames:
    frame.layout = go.Layout(
        annotations=[dict(
            x=0.2, y=0.5, xref="paper", yref="paper",
            text=f"<b>{frame.name}</b>",
            showarrow=False, font=dict(size=28, color="black"),
            xanchor="left", yanchor="middle",
        )]
    )

annee_i = df_final_mg['annee'].min()
df_i = df_extremes[df_extremes['annee'] == annee_i]
fig.add_trace(go.Choroplethmapbox(
    geojson={'type': 'FeatureCollection', 'features': [f for f in geojson['features'] if f['properties']['code'] in df_i['departement'].tolist()]},
    locations=df_i['departement'], z=df_i['densite'], featureidkey='properties.code',
    colorscale=['#E60F05', '#FF6B6B', '#FF9999', '#A4C2F4', '#0E4099', '#05204D'],
    zmin=27.6, zmax=120.41, # Échelle figée pour l'initialisation
    showscale=False, marker=dict(line=dict(width=1.5, color='black'), opacity=1.0), hoverinfo='skip'
))

fig.show(config={"scrollZoom": True})

In [ ]:
annee_cible = 2036

df_cible = (
    df_final_mg[df_final_mg["annee"] == annee_cible]
    .groupby(["departement", "libelle_departement"], as_index=False)["densite"]
    .mean()
    .sort_values("densite")
)

top3_moins = df_cible.head(3)
top3_plus = df_cible.tail(3)

print(f"\nLes 3 départements les MOINS bien pourvus en {annee_cible} :")
print(top3_moins[["libelle_departement", "densite"]].to_string(index=False))

print(f"\nLes 3 départements les MIEUX pourvus en {annee_cible} :")
print(top3_plus[["libelle_departement", "densite"]].to_string(index=False))

In [ ]:

annee_cible = 2024

df_cible = (
    df_densite[
        (df_densite["profession_sante"] == "Médecins généralistes (hors médecins à expertise particulière - MEP)") &
        (df_densite["annee"] == annee_cible)
    ]
    .groupby(["departement", "libelle_departement"], as_index=False)["densite"].mean()
    .sort_values("densite")
)

top3_moins = df_cible.head(3)
top3_plus  = df_cible.tail(3)

print("Les 3 départements les MOINS bien pourvus :")
print(top3_moins[["libelle_departement", "densite"]].to_string(index=False))

print("Les 3 départements les MIEUX pourvus :")
print(top3_plus[["libelle_departement", "densite"]].to_string(index=False))

In [ ]:
print(df_map_mg[df_map_mg["annee"] == 2024]["densite"].describe())

**Remarque :forte inégalité territoriale**


min = 44.100000 médecins pour 100K hab ==>> Seine-Saint-Denis


max = 125.300000 médecins pour 100K hab ==>> Hautes-Alpes



In [ ]:
# TOP ET FLOP
df_2024 = df_map_mg[df_map_mg["annee"] == 2024]

print("Minimum :")
print(df_2024.loc[df_2024["densite"].idxmin(), ["libelle_departement", "densite"]])

print("\nMaximum :")
print(df_2024.loc[df_2024["densite"].idxmax(), ["libelle_departement", "densite"]])

"TOP 3 & FLOP 3 Departement — 2024"

In [ ]:
# ============================================================
# TOP 3 & FLOP 3 DÉPARTEMENTS — 2024
# ============================================================

df_2024 = df_map_mg[df_map_mg["annee"] == 2024].sort_values("densite")

flop3 = df_2024.head(3).assign(categorie="Flop 3")
top3  = df_2024.tail(3).iloc[::-1].assign(categorie="Top 3")

df_plot = pd.concat([top3, flop3])

colors = ["#2ecc71"] * 3 + ["#e74c3c"] * 3

fig = go.Figure(go.Bar(
    x=df_plot["libelle_departement"],
    y=df_plot["densite"],
    marker_color=colors,
    text=df_plot["densite"].round(1),
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Densité : %{y:.1f} pour 100K hab<extra></extra>"
))

fig.update_layout(
    title="Top 3 & Flop 3 départements — Densité médecins généralistes (2024)",
    yaxis_title="Densité (pour 100 000 hab)",
    xaxis_title="Département",
    height=450,
    yaxis=dict(range=[0, df_plot["densite"].max() * 1.2]),
    showlegend=False
)

fig.show()

In [ ]:
# ============================================================
# LINE CHART — TOP 3 & FLOP 3 historique (2010-2024) + prédictions (2025-2036)
# ============================================================

# Fusionner historique + prédictions (toutes les années)
df_complet = pd.concat([df_map_mg, df_pred_mg], ignore_index=True).sort_values("annee")
df_complet = df_complet[df_complet["annee"] <= 2036]  # ← ajoute cette ligne
df_complet = df_complet[~df_complet["departement"].isin(["971", "972", "973", "974", "975", "976"])]

# Top 3 & Flop 3 basés sur 2024
df_ref = df_map_mg[df_map_mg["annee"] == 2024].sort_values("densite")
flop3_depts = df_ref.head(3)["departement"].tolist()
top3_depts  = df_ref.tail(3)["departement"].tolist()

# Filtrer les 6 départements sur toute la période
df_line = df_complet[df_complet["departement"].isin(top3_depts + flop3_depts)].copy()
df_line["categorie"] = df_line["departement"].apply(
    lambda x: "Top 3" if x in top3_depts else "Flop 3"

)
df_line["type"] = df_line["annee"].apply(
    lambda x: "Historique" if x <= 2024 else "Prédiction"
)

# Palette de couleurs
noms_top3  = df_ref.tail(3)["libelle_departement"].tolist()
noms_flop3 = df_ref.head(3)["libelle_departement"].tolist()

color_map = {
    noms_top3[0]:  "#82e0aa",
    noms_top3[1]:  "#2ecc71",
    noms_top3[2]:  "#1a8a4a",
    noms_flop3[0]: "#f1948a",
    noms_flop3[1]: "#e74c3c",
    noms_flop3[2]: "#922b21",
}

fig = px.line(
    df_line,
    x="annee",
    y="densite",
    color="libelle_departement",
    color_discrete_map=color_map,
    markers=True,
    title="Top 3 & Flop 3 — Évolution + Prédictions densité médecins généralistes (2010–2036)",
    labels={
        "annee": "Année",
        "densite": "Densité (pour 100 000 hab)",
        "libelle_departement": "Département"
    }
)

# Annotation "Historique" à gauche
fig.add_annotation(
    x=2017, y=df_line["densite"].max() * 1.05,
    text="← Historique",
    showarrow=False,
    font=dict(size=12, color="gray")
)

# Labels au bout de chaque ligne (à droite)
for dept in df_line["libelle_departement"].unique():
    df_dept = df_line[df_line["libelle_departement"] == dept].sort_values("annee")
    derniere = df_dept.iloc[-1]
    fig.add_annotation(
        x=derniere["annee"] + 0.2,
        y=derniere["densite"],
        text=f"<b>{dept}</b>",
        showarrow=False,
        xanchor="left",
        font=dict(size=11, color=color_map.get(dept, "black"))
    )


fig.update_layout(
    height=520,
    hovermode="x unified",
    showlegend=False,          # labels directs sur le graphe → légende inutile
    margin=dict(r=220),        # espace à droite pour les labels
    xaxis=dict(dtick=2),
)

fig.show()

In [ ]:
DROM = ["971", "972", "973", "974", "975", "976", "999"]

df_densite_drom = client.query("""
SELECT *
FROM `zinebakninsql.Teou.densite`
WHERE CAST(departement AS STRING) IN ('971', '972', '973', '974', '975', '976', '999')
""").to_dataframe()

In [ ]:
df_densite_drom

In [ ]:
print(df_densite_drom[df_densite_drom["annee"] == 2024][["departement", "libelle_departement"]].drop_duplicates().to_string(index=False))

In [4]:
!jupyter nbconvert --to script ton_notebook.ipynb

[NbConvertApp] WARNING | pattern 'ton_notebook.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
  